在基于dqn的基础上设计madqn将各个agent协同完成任务时，发现原作的一个问题。
先看原作的算法：

<img src="figs/image.png" style="width:50%;">

这个算法有一个问题，就是所有的agent采用的都是各自不同的神经网络，也就是说madqn只是将dqn的数量增加，没有互相之间的协同，只是各自完成各自的任务，只要不发生碰撞就行。
但是这样子真的是madqn吗？为什么不能把所有agent的state进行拼接，组合为states vector再训练呢？
调试一下组合states再embedding的方法。

In [2]:
import sys
import os

sys.argv = ['']
sys.path.append("..")

from env.env import Env
from rl_algorithms.agent import Agent
from rl_algorithms.dqn import Qnet, ReplayBuffer

from tqdm import tqdm
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn
import torch
from typing import List, Tuple, Any, Optional
from collections import deque
import random
import numpy as np

In [ ]:
from turtle import forward


class Qnet(nn.Module):
    """
    输入：
        多agent的state，每个agent的state idx按照顺序拼接为state idx vector
        所有agent的target，target不分先后顺序，是所有agent协同配合的target
    输出：
        当前多agent的action value集合
    target 是常数，不作为网络输入
    相对位置信息作为主干
    """
    def __init__(self, states, targets, action_dim, hidden_dim) -> None:
        super().__init__()
        self.states = states
        self.targets = targets
        
        self.states_dim = len(states)
        
        self.embedding = nn.Embedding()
    